# nb00b - Extract: Medi-Cal Managed Care Provider Listing (for the access-gap view)

**Purpose:** pull the CHHS **Medi-Cal Managed Care Provider Listing** so we can count providers by county, by plan, and by provider type, and then combine that with enrollment to measure **network adequacy** (providers per member): where members are covered but may lack nearby providers who accept the plan.

**Dataset:** https://data.chhs.ca.gov/dataset/medi-cal-managed-care-provider-listing
Provider level records with plan, county, address, and provider type (each MCP submits one file per county).

**Approach:** this notebook is deliberately cautious. It first lists every resource and their sizes, then pulls the CSV, then profiles the columns, the plans, the counties, and the provider types **before** we decide how to aggregate. A statewide provider file can be large, so we inspect before committing.

**Pipeline:** nb00b (extract, this file) -> nb01b or a new clean step -> Tableau access-gap sheet.

In [1]:
import json
from datetime import date
from pathlib import Path
import pandas as pd
import requests

# The provider listing is published on both open-data portals; we try both.
PORTALS = ['https://data.chhs.ca.gov', 'https://data.ca.gov']
DATASET_SLUG = 'medi-cal-managed-care-provider-listing'

DATA = Path('..') / 'data'
RAW = DATA / 'raw'
RAW.mkdir(parents=True, exist_ok=True)
print('raw folder:', RAW.resolve())

raw folder: /Users/trinidadcisneros/Documents/Development/trinidadcisneros.github.io/folders/ds_blogs/projects/tableau/tableau_la_market_share/data/raw


## 1. List every resource in the dataset (look before pulling)

The provider listing may be one consolidated CSV, or many files. Print them all with format and size so we know what we are about to download.

In [2]:
def resolve_dataset(slug):
    """Try each portal and each id form; return (portal, package) or raise with a clear message."""
    tried = []
    for base in PORTALS:
        for pid in (slug, slug.replace('-', '_')):
            url = f'{base}/api/3/action/package_show'
            try:
                r = requests.get(url, params={'id': pid}, timeout=60)
                j = r.json()
            except Exception as e:
                tried.append(f'{base} id={pid}: request/parse failed ({e})')
                continue
            if isinstance(j, dict) and j.get('success') and 'result' in j:
                print(f'FOUND on {base}  (id={pid})')
                return base, j['result']
            # not success: capture the error message CKAN returned
            err = j.get('error') if isinstance(j, dict) else None
            tried.append(f'{base} id={pid}: success={j.get("success") if isinstance(j,dict) else "?"} error={err}')
    raise RuntimeError('Could not resolve the dataset. Attempts:\n  ' + '\n  '.join(tried))

PORTAL, pkg = resolve_dataset(DATASET_SLUG)
print('Dataset:', pkg.get('title'))
print('Resources:', len(pkg['resources']))
print()
for i, r in enumerate(pkg['resources']):
    size = r.get('size')
    size_mb = f'{int(size)/1_048_576:.1f} MB' if size else 'unknown size'
    print(f"[{i}] {r.get('format','?'):8} | {size_mb:>12} | {r.get('name','')[:60]}")
    print(f'      {r.get("url","")}')

csv_res = [r for r in pkg['resources'] if r.get('format','').upper() == 'CSV']
print(f'\nCSV resources: {len(csv_res)}')

FOUND on https://data.ca.gov  (id=medi-cal-managed-care-provider-listing)
Dataset: Medi-Cal Managed Care Provider Listing
Resources: 4

[0] HTML     | unknown size | ArcGIS Hub Dataset
      https://gis.dhcs.ca.gov/datasets/CADHCS::medi-cal-managed-care-provider-listing/explore?location=37.261272%2C-99.127482%2C4
[1] arcgis geoservices rest api | unknown size | ArcGIS GeoService
      https://services7.arcgis.com/7MUwsS9z05YumJRZ/arcgis/rest/services/Medi_Cal_MC_Provider_Listing/FeatureServer/0
[2] CSV      | unknown size | CSV
      https://data.chhs.ca.gov/dataset/77c2ec7a-b421-4ffc-8842-13caf384bea7/resource/da5eee46-b040-45cc-8f92-3a7601342cf6/download/mc_providerlist_0725.csv
[3] ZIP      | unknown size | All resource data
      https://data.chhs.ca.gov/dataset/77c2ec7a-b421-4ffc-8842-13caf384bea7/resource/2142138a-984b-4431-9c3b-610bfc099bce/download/medi-cal-mc-provider-listing-r2kn_m1x.zip

CSV resources: 1


## 2. Pull the CSV provider listing

If there is a single consolidated CSV, this grabs it. If there are several, it grabs the largest (usually the full listing). Watch the size printed above; if it is very large (over ~500 MB), stop and tell Claude, and we will pull a filtered slice instead.

In [3]:
# choose the largest CSV resource (typically the full provider listing)
resource = max(csv_res, key=lambda r: int(r.get('size') or 0)) if csv_res else None
assert resource is not None, 'no CSV resource found; check the printed list above'
print('pulling:', resource.get('name'), '\n', resource['url'])

dl = requests.get(resource['url'], timeout=1200)
dl.raise_for_status()
raw_path = RAW / f"mc_provider_listing_raw_{date.today().isoformat()}.csv"
raw_path.write_bytes(dl.content)
print(f'saved {raw_path.name}  ({raw_path.stat().st_size/1_048_576:.1f} MB)')

(DATA / 'provider_extraction_log.json').write_text(json.dumps({
    'dataset': pkg['title'], 'resource': resource.get('name'),
    'url': resource['url'], 'downloaded': date.today().isoformat(),
    'raw_file': raw_path.name,
}, indent=2))

pulling: CSV 
 https://data.chhs.ca.gov/dataset/77c2ec7a-b421-4ffc-8842-13caf384bea7/resource/da5eee46-b040-45cc-8f92-3a7601342cf6/download/mc_providerlist_0725.csv
saved mc_provider_listing_raw_2026-07-22.csv  (1511.4 MB)


326

## 3. Profile: columns, plans, counties, provider types

This tells us which columns identify the plan, the county, and the provider category, so we can design the access metric. Read low memory off to avoid dtype warnings on a wide file.

In [4]:
prov = pd.read_csv(raw_path, dtype=str, low_memory=False)
print('shape:', prov.shape)
print('\ncolumns:')
for c in prov.columns:
    print('  ', c)

# best-guess key columns (names vary; adjust after seeing the list above)
def find_col(cols, *keys):
    for c in cols:
        cl = c.lower()
        if all(k in cl for k in keys):
            return c
    return None

col_plan   = find_col(prov.columns, 'plan')
col_county = find_col(prov.columns, 'county')
col_type   = find_col(prov.columns, 'provider', 'type') or find_col(prov.columns, 'category')

print('\nguessed key columns -> plan:', col_plan, '| county:', col_county, '| type:', col_type)
for label, c in [('plans', col_plan), ('counties', col_county), ('provider types', col_type)]:
    if c:
        vals = prov[c].dropna().unique()
        print(f'\n{label} ({len(vals)}):', sorted(vals)[:25])

shape: (5043168, 30)

columns:
   ManagedCarePlan
   SubNetwork
   PlanCode
   RecordType
   NPI
   FacilityName
   LastName
   FirstName
   Taxonomy
   MCNAProviderGroup
   MCNAProviderType
   FacilityType
   LicensureType
   PrimaryCare
   Specialist
   SeesChildren
   Telehealth
   BHIndicator
   Address
   Address2
   City
   State
   Zip
   Zip4
   TelephoneNumber
   Longitude
   Latitidue
   DHCSCountyCode
   FIPSCd
   County

guessed key columns -> plan: ManagedCarePlan | county: DHCSCountyCode | type: MCNAProviderType

plans (24): ['AIDS Healthcare Foundation', 'ANTHEM BLUE CROSS PARTNERSHIP PLAN', 'Alameda Alliance for Health', 'Blue Shield of California Promise', 'CENCAL HEALTH', 'CENTRAL CALIFORNIA ALLIANCE FOR HEALTH', 'CONTRA COSTA HEALTH PLAN', 'CalOptima', 'CalViva', 'Community Health Group Partnership Plan', 'Community Health Plan of Imperial Valley', 'GOLD COAST HEALTH PLAN', 'Health Net Community Solutions Inc.', 'Health Plan of San Joaquin', 'Health Plan of San Mateo

## 4. Los Angeles focus (the market this project covers)

Once the county column is confirmed, look at Los Angeles: which plans have providers listed, and how many providers by type. This is the raw material for the access-gap metric (providers per member).

In [5]:
if col_county and col_plan:
    la = prov[prov[col_county].str.contains('Los Angeles', case=False, na=False)]
    print('LA provider rows:', len(la))
    print('\nLA providers by plan:')
    print(la[col_plan].value_counts().head(15).to_string())
    if col_type:
        print('\nLA providers by type (top 15):')
        print(la[col_type].value_counts().head(15).to_string())
else:
    print('confirm the county/plan column names from the printed columns, then rerun this cell')

LA provider rows: 0

LA providers by plan:
Series([], )

LA providers by type (top 15):
Series([], )
